# Batch 02 — Fixed Methodology Benchmark

**What changed from Batch 01:**
- Frames pre-buffered into RAM before the timed loop (I/O decoupled from inference)
- First 20 warmup frames excluded from all statistics
- ONNX confirmed on CUDA Execution Provider
- `torch.compile` added as a new runtime
- All raw timings saved for every runtime (no missing FP16 raw data)
- Output files prefixed with `b02_` to avoid name collisions with Batch 01

Run on **Google Colab with a T4 GPU** (Runtime → Change runtime type → T4 GPU).

## 0. Configuration

In [ ]:
REPO_URL         = 'https://github.com/khushalrs/Inference_pipeline_eval.git'
REPO_DIR         = 'Inference_pipeline_eval'
DRIVE_VIDEO_PATH = '/content/drive/MyDrive/Colab Notebooks/car_clip.mp4'

import os
WORKSPACE = f'/content/{REPO_DIR}'
print(f'Workspace : {WORKSPACE}')

## 1. Setup

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
import os

if os.path.exists(WORKSPACE):
    print('Repo already cloned — pulling latest changes...')
    !git -C {WORKSPACE} pull
else:
    !git clone {REPO_URL} {WORKSPACE}

%cd {WORKSPACE}
!pwd

In [ ]:
!pip install -q -r requirements.txt
# Force GPU-capable ORT — plain onnxruntime will silently fall back to CPU EP
!pip install onnxruntime-gpu==1.20.1 --force-reinstall -q

## 2. Environment Check

In [ ]:
import torch
import onnxruntime as ort

print(f'PyTorch version      : {torch.__version__}')
print(f'torch.compile support: {hasattr(torch, "compile")}')
print(f'CUDA available       : {torch.cuda.is_available()}')
if torch.cuda.is_available():
    print(f'GPU                  : {torch.cuda.get_device_name(0)}')
    print(f'CUDA version         : {torch.version.cuda}')
    print(f'VRAM                 : {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB')
print(f'ORT available EPs    : {ort.get_available_providers()}')
assert 'CUDAExecutionProvider' in ort.get_available_providers(), \
    'CUDAExecutionProvider missing — reinstall onnxruntime-gpu'

## 3. Copy Video from Drive

In [ ]:
import shutil, os

dest = 'data/clip.mp4'
os.makedirs('data',    exist_ok=True)
os.makedirs('results', exist_ok=True)
os.makedirs('plots',   exist_ok=True)
os.makedirs('models',  exist_ok=True)

if not os.path.exists(dest):
    print(f'Copying {DRIVE_VIDEO_PATH} → {dest} ...')
    shutil.copy(DRIVE_VIDEO_PATH, dest)
    print('Done.')
else:
    print(f'{dest} already present.')

print(f'File size: {os.path.getsize(dest)/1e6:.1f} MB')

## 4. Export ONNX Model

Only needs to run once per session.

In [ ]:
!python3 scripts/export_onnx.py \
    --model      yolo11n.pt \
    --output     models/yolo11n.onnx \
    --input-size 640 \
    --validate \
    --video      data/clip.mp4 \
    --val-frames 20 \
    --device     cuda

## 5. Build TensorRT Engines

Build time ~5–10 min total. Must rebuild for every new Colab session.

In [ ]:
!pip install tensorrt-cu12 --extra-index-url https://pypi.nvidia.com -q

In [ ]:
!python3 scripts/build_tensorrt_engine.py \
    --onnx         models/yolo11n.onnx \
    --output-dir   models/ \
    --fp32 --fp16 \
    --workspace-gb 2

---
## 6. Run All Runtimes

Run each cell in order. Each script:
- Pre-buffers all frames into RAM before timing starts (I/O excluded)
- Warms up for 20 iterations before recording
- Saves per-frame raw timings CSV

Stages timed: `preprocess → inference → postprocess` (no disk read, no draw)

### Runtime 1 — PyTorch Eager

In [ ]:
!python3 scripts/run_pytorch_video.py \
    --video      data/clip.mp4 \
    --results    results/b02_pytorch_raw_timings.csv \
    --model      yolo11n.pt \
    --input-size 640 \
    --conf       0.25 \
    --iou        0.45 \
    --warmup     20 \
    --device     cuda

In [ ]:
!python3 scripts/benchmark.py \
    --results       results/b02_pytorch_raw_timings.csv \
    --output        results/b02_pytorch_benchmark.csv \
    --runtime       pytorch \
    --warmup-frames 20

### Runtime 2 — torch.compile (reduce-overhead mode)

In [ ]:
!python3 scripts/run_compile_video.py \
    --video      data/clip.mp4 \
    --results    results/b02_compile_raw_timings.csv \
    --model      yolo11n.pt \
    --input-size 640 \
    --conf       0.25 \
    --iou        0.45 \
    --warmup     20 \
    --device     cuda

In [ ]:
!python3 scripts/benchmark.py \
    --results       results/b02_compile_raw_timings.csv \
    --output        results/b02_compile_benchmark.csv \
    --runtime       compile \
    --warmup-frames 20

### Runtime 3 — ONNX Runtime (CUDA Execution Provider)

In [ ]:
!python3 scripts/run_onnx_video.py \
    --video      data/clip.mp4 \
    --onnx-model models/yolo11n.onnx \
    --results    results/b02_onnx_cuda_raw_timings.csv \
    --input-size 640 \
    --conf       0.25 \
    --iou        0.45 \
    --warmup     20 \
    --device     cuda

In [ ]:
!python3 scripts/benchmark.py \
    --results       results/b02_onnx_cuda_raw_timings.csv \
    --output        results/b02_onnx_cuda_benchmark.csv \
    --runtime       onnx-cuda \
    --warmup-frames 20

### Runtime 4 — TensorRT FP32

In [ ]:
!python3 scripts/run_tensorrt_video.py \
    --video      data/clip.mp4 \
    --engine     models/yolo11n_fp32.engine \
    --results    results/b02_trt_fp32_raw_timings.csv \
    --input-size 640 \
    --conf       0.25 \
    --iou        0.45 \
    --warmup     20

In [ ]:
!python3 scripts/benchmark.py \
    --results       results/b02_trt_fp32_raw_timings.csv \
    --output        results/b02_trt_fp32_benchmark.csv \
    --runtime       trt-fp32 \
    --warmup-frames 20

### Runtime 5 — TensorRT FP16

In [ ]:
!python3 scripts/run_tensorrt_video.py \
    --video      data/clip.mp4 \
    --engine     models/yolo11n_fp16.engine \
    --results    results/b02_trt_fp16_raw_timings.csv \
    --input-size 640 \
    --conf       0.25 \
    --iou        0.45 \
    --warmup     20

In [ ]:
!python3 scripts/benchmark.py \
    --results       results/b02_trt_fp16_raw_timings.csv \
    --output        results/b02_trt_fp16_benchmark.csv \
    --runtime       trt-fp16 \
    --warmup-frames 20

---
## 7. Generate Comparison Charts

All 5 runtimes in one set of charts. Prefixed with `b02_` so they don't overwrite Batch 01 plots.

In [ ]:
!python3 scripts/plot_results.py \
    --baselines results/b02_pytorch_benchmark.csv \
                results/b02_compile_benchmark.csv \
                results/b02_onnx_cuda_benchmark.csv \
                results/b02_trt_fp32_benchmark.csv \
                results/b02_trt_fp16_benchmark.csv \
    --labels    pytorch compile onnx-cuda trt-fp32 trt-fp16 \
    --output-dir plots/ \
    --prefix    b02_

In [ ]:
from IPython.display import Image, display

for chart in ['fps_comparison', 'pipeline_breakdown', 'latency_percentiles']:
    print(f'── {chart} ──')
    display(Image(filename=f'plots/b02_{chart}.png'))

---
## 8. Download All Results

Download everything before the session expires.

In [ ]:
from google.colab import files

raw_timings = [
    'results/b02_pytorch_raw_timings.csv',
    'results/b02_compile_raw_timings.csv',
    'results/b02_onnx_cuda_raw_timings.csv',
    'results/b02_trt_fp32_raw_timings.csv',
    'results/b02_trt_fp16_raw_timings.csv',
]

benchmarks = [
    'results/b02_pytorch_benchmark.csv',
    'results/b02_compile_benchmark.csv',
    'results/b02_onnx_cuda_benchmark.csv',
    'results/b02_trt_fp32_benchmark.csv',
    'results/b02_trt_fp16_benchmark.csv',
]

charts = [
    'plots/b02_fps_comparison.png',
    'plots/b02_pipeline_breakdown.png',
    'plots/b02_latency_percentiles.png',
]

for f in raw_timings + benchmarks + charts:
    print(f'Downloading {f} ...')
    files.download(f)